# New Analysis for Appeal

## Setup + helper functions

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.cov_struct import Independence, Exchangeable
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_1samp, binomtest, norm

warnings.filterwarnings("ignore")

# ----------------------------
# Paths
# ----------------------------
DATA_PATH   = "../../cnt-data/"
TABLES_PATH = os.path.join(DATA_PATH, "revised_tables_step_4")
SOZ_PATH    = os.path.join(DATA_PATH, "HUP_SOZ.csv")

OUT_BASE = os.path.join(DATA_PATH, "pi_notebook_new_analyses")
os.makedirs(OUT_BASE, exist_ok=True)

# ----------------------------
# Model spec
# ----------------------------
Y = "med_sum_no_lorazepam_raw"
GROUP = "patient_hup_id"
TIME_RAW = "emu_minute_raw"
SYNC = "synchrony_broadband"

PREDICTORS = [
    "synchrony_broadband",
    "time_since_last_seizure",
    "time_until_next_seizure",
    "emu_minute",
    "ad_ratio",
    "tod_sin",
    "tod_cos",
    "tod_sin2",
    "tod_cos2",
    "spike_rate",
]

COLS_TO_KEEP = [Y, GROUP, "emu_minute"] + PREDICTORS  # GROUP repeated ok
COLS_TO_SCALE = [
    "synchrony_broadband",
    "time_since_last_seizure",
    "time_until_next_seizure",
    "emu_minute",
    "ad_ratio",
    "spike_rate",
]

FORMULA = f"{Y} ~ " + " + ".join(PREDICTORS)

# ----------------------------
# Utilities
# ----------------------------
def get_patient_hup_ids(directory):
    files = os.listdir(directory)
    ids = [int(f.split("_")[1].split(".")[0]) for f in files if f.startswith("HUP_") and f.endswith(".csv")]
    return sorted(ids)

def dedupe_columns(df):
    # keep first occurrence if duplicates exist
    return df.loc[:, ~df.columns.duplicated()].copy()

def load_patient_table(pid):
    f = os.path.join(TABLES_PATH, f"HUP_{pid}.csv")
    df = pd.read_csv(f, low_memory=False)
    df = dedupe_columns(df)
    df[GROUP] = pid
    # keep only columns present
    keep = [c for c in COLS_TO_KEEP if c in df.columns]
    df = df[keep].copy()
    return df

def cohort_patient_ids():
    completed = get_patient_hup_ids(TABLES_PATH)

    soz = pd.read_csv(SOZ_PATH)
    mtle = soz[soz["SOZ"] == "mesial temporal"].copy()
    mtle["pt_id"] = mtle["pt_id"].astype(str).str.replace("HUP", "", regex=False).astype(int)
    mtle_ids = sorted(mtle["pt_id"].unique().tolist())

    mtle_completed = sorted([p for p in completed if p in mtle_ids])
    non_mtle_completed = sorted([p for p in completed if p not in mtle_ids])

    return completed, mtle_completed, non_mtle_completed

def build_all_data(pids):
    dfs = []
    for pid in pids:
        fp = os.path.join(TABLES_PATH, f"HUP_{pid}.csv")
        if not os.path.exists(fp):
            continue
        d = pd.read_csv(fp, low_memory=False)
        d[GROUP] = pid
        # Keep only columns that exist, but don't try to dedupe here yet.
        keep = [c for c in ([GROUP, "emu_minute"] + PREDICTORS + [Y]) if c in d.columns]
        dfs.append(d[keep])

    if len(dfs) == 0:
        raise ValueError("No tables loaded for requested patient IDs.")

    all_data = pd.concat(dfs, axis=0, ignore_index=True)

    # ---- FIX 1: drop duplicated column labels (keeps first occurrence) ----
    if all_data.columns.duplicated().any():
        dupes = all_data.columns[all_data.columns.duplicated()].unique().tolist()
        print(f"[build_all_data] Warning: duplicated columns found and dropped: {dupes}")
        all_data = all_data.loc[:, ~all_data.columns.duplicated()].copy()

    # ---- FIX 2: guarantee emu_minute is a single numeric Series ----
    if "emu_minute" not in all_data.columns:
        raise ValueError("Expected 'emu_minute' in tables but it is missing after load.")

    all_data["emu_minute"] = pd.to_numeric(all_data["emu_minute"], errors="coerce")

    # raw time copy before scaling
    all_data[TIME_RAW] = all_data["emu_minute"].copy()

    # drop NA on needed columns
    needed = [Y, GROUP, "emu_minute", TIME_RAW] + [c for c in PREDICTORS if c in all_data.columns]
    all_data = all_data.dropna(subset=needed).reset_index(drop=True)

    # sort and add within-patient index
    all_data = all_data.sort_values([GROUP, "emu_minute"]).reset_index(drop=True)
    all_data["time_index"] = all_data.groupby(GROUP).cumcount().astype(int)

    # scale selected predictors
    scaler = StandardScaler()
    cols_to_scale = [c for c in COLS_TO_SCALE if c in all_data.columns]
    all_data[cols_to_scale] = scaler.fit_transform(all_data[cols_to_scale])

    return all_data, scaler


def bin_within_patient(df, bin_minutes):
    d = df.copy()
    d["bin"] = np.floor(d[TIME_RAW] / float(bin_minutes)).astype(int)
    # mean-aggregate within patient/bin
    agg_cols = [Y] + PREDICTORS + [GROUP, TIME_RAW]
    out = (d[agg_cols + ["bin"]]
           .groupby([GROUP, "bin"], as_index=False)
           .mean(numeric_only=True))
    # rebuild ordering + time_index
    out = out.sort_values([GROUP, TIME_RAW]).reset_index(drop=True)
    out["time_index"] = out.groupby(GROUP).cumcount().astype(int)
    return out

def subsample_every_k(df, k):
    d = df.sort_values([GROUP, TIME_RAW]).copy()
    # take every k-th row within patient
    keep_idx = d.groupby(GROUP).cumcount() % int(k) == 0
    out = d.loc[keep_idx].reset_index(drop=True)
    out["time_index"] = out.groupby(GROUP).cumcount().astype(int)
    return out

def fit_gee(df, cov_struct):
    m = smf.gee(
        formula=FORMULA,
        groups=df[GROUP],
        data=df,
        family=Gaussian(),
        cov_struct=cov_struct,
    )
    r = m.fit(cov_type="robust")
    return r

def fit_mixedlm_random_intercept(df):
    m = smf.mixedlm(FORMULA, df, groups=df[GROUP])
    r = m.fit(method="lbfgs")
    return r

def fit_mixedlm_random_slope(df):
    # random intercept + random slope for synchrony
    m = smf.mixedlm(FORMULA, df, groups=df[GROUP], re_formula="1 + synchrony_broadband")
    r = m.fit(method="lbfgs")
    return r

def patient_adjusted_slopes(df_binned, min_n=20):
    # per-patient OLS with covariates; returns beta(se) for synchrony
    rows = []
    for pid, g in df_binned.groupby(GROUP):
        if len(g) < min_n:
            continue
        try:
            r = smf.ols(FORMULA, data=g).fit()
            beta = float(r.params.get(SYNC, np.nan))
            se = float(r.bse.get(SYNC, np.nan))
            rows.append({"patient_hup_id": int(pid), "n": int(len(g)), "beta": beta, "se": se})
        except Exception:
            continue
    return pd.DataFrame(rows)

def across_patient_tests(slopes_df):
    d = slopes_df.dropna(subset=["beta"]).copy()
    betas = d["beta"].values
    n = len(betas)
    if n == 0:
        return {}
    frac_neg = float(np.mean(betas < 0))
    t_p = float(ttest_1samp(betas, 0.0, nan_policy="omit").pvalue) if n > 1 else np.nan

    # sign test (exclude exact zeros)
    nonzero = betas[betas != 0]
    if len(nonzero) > 0:
        k_neg = int(np.sum(nonzero < 0))
        p_sign = float(binomtest(k_neg, len(nonzero), 0.5, alternative="two-sided").pvalue)
    else:
        p_sign = np.nan

    # inverse-variance weighted meta (requires se>0)
    d2 = d.dropna(subset=["se"]).copy()
    d2 = d2[d2["se"] > 0].copy()
    if len(d2) >= 2:
        w = 1.0 / (d2["se"].values ** 2)
        beta_hat = float(np.sum(w * d2["beta"].values) / np.sum(w))
        se_hat = float(np.sqrt(1.0 / np.sum(w)))
        z = beta_hat / se_hat if se_hat > 0 else np.nan
        p_meta = float(2 * (1 - norm.cdf(abs(z)))) if np.isfinite(z) else np.nan
    else:
        beta_hat, se_hat, p_meta = np.nan, np.nan, np.nan

    return {
        "n_patients_used": int(n),
        "frac_negative": frac_neg,
        "t_test_p": t_p,
        "sign_test_p": p_sign,
        "meta_beta_ivw": beta_hat,
        "meta_se_ivw": se_hat,
        "meta_p_ivw": p_meta,
    }

def lopo_gee(df, cov_struct):
    base = fit_gee(df, cov_struct)
    base_beta = float(base.params[SYNC])
    rows = []
    for pid in sorted(df[GROUP].unique()):
        d = df[df[GROUP] != pid]
        r = fit_gee(d, cov_struct)
        rows.append({
            "left_out": int(pid),
            "beta": float(r.params[SYNC]),
            "se": float(r.bse[SYNC]),
            "p": float(r.pvalues[SYNC]),
            "delta": float(r.params[SYNC] - base_beta),
        })
    out = pd.DataFrame(rows).sort_values("delta", key=lambda s: np.abs(s), ascending=False)
    return base, out

def circular_shift_permutation(df, n_perm=300, seed=0, min_shift=10):
    rng = np.random.default_rng(seed)
    obs = fit_gee(df, Independence())
    obs_beta = float(obs.params[SYNC])

    betas = []
    for b in range(n_perm):
        d = df.copy()
        # within each patient, circularly shift synchrony to break alignment but preserve autocorrelation
        for pid, g_idx in d.groupby(GROUP).groups.items():
            idx = np.array(list(g_idx))
            if len(idx) <= min_shift:
                continue
            vals = d.loc[idx, SYNC].values
            shift = int(rng.integers(min_shift, len(vals)))
            d.loc[idx, SYNC] = np.roll(vals, shift)
        r = fit_gee(d, Independence())
        betas.append(float(r.params[SYNC]))

    betas = np.array(betas)
    p_two_sided = float(np.mean(np.abs(betas) >= abs(obs_beta)))
    return obs_beta, betas, p_two_sided

print("Setup complete.")
print("Tables:", TABLES_PATH)
print("SOZ file:", SOZ_PATH)


Setup complete.
Tables: ../../cnt-data/revised_tables_step_4
SOZ file: ../../cnt-data/HUP_SOZ.csv


## Choose cohort + load data

In [2]:
# Choose: "full", "mtle", "non_mtle"
COHORT = "full"

completed, mtle_completed, non_mtle_completed = cohort_patient_ids()

pids = completed

print(f"Found {len(completed)} patient tables in {TABLES_PATH}")
print(f"mTLE patients with tables: {len(mtle_completed)}")
print(f"non-mTLE patients with tables: {len(non_mtle_completed)}")

df, scaler = build_all_data(pids)

out_dir = os.path.join(OUT_BASE, COHORT)
os.makedirs(out_dir, exist_ok=True)

print(f"\nCOHORT={COHORT}")
print(f"Rows after dropna: {len(df)}")
print(f"Unique patients: {df[GROUP].nunique()}")
print(f"Outputs will save to: {out_dir}")


Found 68 patient tables in ../../cnt-data/revised_tables_step_4
mTLE patients with tables: 21
non-mTLE patients with tables: 47
[build_all_data] Warning: duplicated columns found and dropped: ['emu_minute']

COHORT=full
Rows after dropna: 23402
Unique patients: 67
Outputs will save to: ../../cnt-data/pi_notebook_new_analyses/full


In [3]:
import statsmodels.api as sm
from statsmodels.genmod.cov_struct import Autoregressive

# ----------------------------
# Settings
# ----------------------------
BIN_MINUTES_FOR_AR1 = 30     # set to None to run on unbinned df; 30 is a good stabilizer
AR1_RHO_GRID = [0.99, 0.98, 0.95, 0.90, 0.80]  # try these if estimation fails
PRINT_SUMMARY = True

# ----------------------------
# Prepare data (optional binning)
# ----------------------------
if BIN_MINUTES_FOR_AR1 is None:
    df_ar1 = df.copy()
    label = "unbinned"
else:
    df_ar1 = bin_within_patient(df, BIN_MINUTES_FOR_AR1)
    label = f"binned{BIN_MINUTES_FOR_AR1}min"

print(f"\n====================")
print(f"GEE AR(1) on {COHORT} ({label})")
print(f"Rows: {len(df_ar1)}, Patients: {df_ar1[GROUP].nunique()}")
print(f"====================")

# ----------------------------
# Helper: fit AR1 with estimated rho (may fail in statsmodels)
# ----------------------------
def fit_gee_ar1_estimated(data):
    # time index is already in data["time_index"]
    m = smf.gee(
        formula=FORMULA,
        groups=data[GROUP],
        data=data,
        family=Gaussian(),
        cov_struct=Autoregressive(),
        time=data["time_index"].values,
    )
    r = m.fit(cov_type="robust")
    rho = getattr(r.cov_struct, "dep_params", np.nan)
    return r, "AR1_estimated", float(rho) if np.ndim(rho) == 0 else float(np.asarray(rho).ravel()[0])

# ----------------------------
# Helper: fit AR1 with fixed rho (manual hack via subclass, compatible across versions)
# ----------------------------
class FixedAR1(Autoregressive):
    def __init__(self, rho):
        super().__init__()
        self._fixed_rho = float(rho)

    def update(self, params):
        # lock rho; statsmodels will use this for the working correlation
        self.dep_params = self._fixed_rho

def fit_gee_ar1_fixed(data, rho):
    m = smf.gee(
        formula=FORMULA,
        groups=data[GROUP],
        data=data,
        family=Gaussian(),
        cov_struct=FixedAR1(rho),
        time=data["time_index"].values,
    )
    r = m.fit(cov_type="robust")
    return r, f"AR1_fixed(rho={rho})", float(rho)

# ----------------------------
# Try estimated AR1 first; if it fails, fall back to fixed rho grid
# ----------------------------
mode = None
rho_used = np.nan
res = None
err = None

try:
    res, mode, rho_used = fit_gee_ar1_estimated(df_ar1)
except Exception as e:
    err = e
    print("\nAR(1) rho estimation failed:")
    print(" ", repr(e))
    print("\nTrying fixed-rho fallback grid...")
    for rho in AR1_RHO_GRID:
        try:
            res, mode, rho_used = fit_gee_ar1_fixed(df_ar1, rho)
            break
        except Exception as e2:
            continue

if res is None:
    raise RuntimeError(
        "GEE AR(1) failed for both estimated and fixed-rho attempts. "
        "Try increasing BIN_MINUTES_FOR_AR1 (e.g., 60) or reducing AR1_RHO_GRID."
    )

print(f"\nGEE AR(1) mode: {mode}, rho_used={rho_used}")

# ----------------------------
# Display / save results
# ----------------------------
if PRINT_SUMMARY:
    print(res.summary())

# key stats for synchrony
beta_sync = float(res.params[SYNC])
se_sync = float(res.bse[SYNC])
p_sync = float(res.pvalues[SYNC])

print(f"\nSynchrony ({SYNC}) under {mode}:")
print(f"  beta = {beta_sync:.6f}")
print(f"  se   = {se_sync:.6f}")
print(f"  p    = {p_sync:.6g}")

# Save outputs
out_txt = os.path.join(out_dir, f"gee_ar1_{label}_summary.txt")
out_csv = os.path.join(out_dir, f"gee_ar1_{label}_params.csv")

with open(out_txt, "w") as f:
    f.write(f"COHORT={COHORT}\n")
    f.write(f"label={label}\n")
    f.write(f"mode={mode}\n")
    f.write(f"rho_used={rho_used}\n\n")
    f.write(res.summary().as_text())
    if err is not None:
        f.write("\n\nAR1 estimation error:\n")
        f.write(repr(err))

pd.DataFrame({
    "term": res.params.index,
    "beta": res.params.values,
    "se": res.bse.values,
    "p": res.pvalues.values
}).to_csv(out_csv, index=False)

print(f"\nSaved:")
print(f"  {out_txt}")
print(f"  {out_csv}")



GEE AR(1) on full (binned30min)
Rows: 7900, Patients: 67

AR(1) rho estimation failed:
  ValueError('Autoregressive: unable to find right bracket')

Trying fixed-rho fallback grid...

GEE AR(1) mode: AR1_fixed(rho=0.99), rho_used=0.99
                                 GEE Regression Results                                
Dep. Variable:        med_sum_no_lorazepam_raw   No. Observations:                 7900
Model:                                     GEE   No. clusters:                       67
Method:                            Generalized   Min. cluster size:                   1
                          Estimating Equations   Max. cluster size:                 378
Family:                               Gaussian   Mean cluster size:               117.9
Dependence structure:                 FixedAR1   Num. iterations:                     3
Date:                         Thu, 09 Apr 2026   Scale:                           2.812
Covariance type:                        robust   Time:      

In [4]:
# Cell — ARIMA-style autocorrelation control: per-patient ARIMAX (AR(1)) + across-patient aggregation

import statsmodels.api as sm

# ----------------------------
# Settings
# ----------------------------
BIN_MINUTES_ARIMAX = 30     # bin first so AR(1) is meaningful (roughly equal spacing) + faster fits
MIN_N_PER_PATIENT  = 40     # minimum binned timepoints needed to fit AR(1) + multiple covariates
MAXITER            = 200

# ----------------------------
# Pick ARIMA engine (prefer SARIMAX)
# ----------------------------
use_sarimax = True
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
except Exception:
    use_sarimax = False
    from statsmodels.tsa.arima.model import ARIMA

# ----------------------------
# Prepare data
# ----------------------------
d = bin_within_patient(df, BIN_MINUTES_ARIMAX)
d = d.sort_values([GROUP, TIME_RAW]).reset_index(drop=True)

# Keep only needed cols & drop NA
needed = [Y, GROUP, TIME_RAW] + PREDICTORS
d = d.dropna(subset=needed).copy()

print("\n====================")
print(f"Per-patient ARIMAX(AR1) on COHORT={COHORT}")
print(f"Data: {BIN_MINUTES_ARIMAX}-min binned")
print(f"Rows: {len(d)}  Patients: {d[GROUP].nunique()}")
print("====================\n")

# ----------------------------
# Fit per patient
# ----------------------------
rows = []
n_fail = 0

for pid, g in d.groupby(GROUP):
    g = g.sort_values(TIME_RAW).copy()
    if len(g) < MIN_N_PER_PATIENT:
        continue

    y = g[Y].astype(float)

    # Exog: all predictors (includes synchrony + covariates)
    X = g[PREDICTORS].astype(float)

    try:
        if use_sarimax:
            # AR(1) residual via SARIMAX with exogenous regressors
            m = SARIMAX(
                endog=y,
                exog=X,
                order=(1, 0, 0),
                trend="c",  # intercept
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            r = m.fit(disp=False, method="lbfgs", maxiter=MAXITER)
        else:
            # Fallback: ARIMA with exog (if SARIMAX not available)
            m = ARIMA(endog=y, exog=X, order=(1, 0, 0), trend="c")
            r = m.fit(method_kwargs={"maxiter": MAXITER})

        params = r.params
        bse = r.bse

        beta = float(params.get(SYNC, np.nan))
        se   = float(bse.get(SYNC, np.nan))

        # AR(1) coefficient name differs by engine
        phi = np.nan
        for nm in ["ar.L1", "arparams", "phi", "ar1"]:
            if nm in params.index:
                phi = float(params[nm])
                break
        # SARIMAX uses "ar.L1"
        if np.isnan(phi) and "ar.L1" in getattr(params, "index", []):
            phi = float(params["ar.L1"])

        rows.append({
            "patient_hup_id": int(pid),
            "n": int(len(g)),
            "beta": beta,
            "se": se,
            "phi_ar1": phi,
            "engine": "SARIMAX" if use_sarimax else "ARIMA",
        })

    except Exception:
        n_fail += 1
        continue

arimax_df = pd.DataFrame(rows)

print(f"Fits succeeded: {len(arimax_df)}  |  failed: {n_fail}")
if len(arimax_df) == 0:
    raise RuntimeError("No per-patient ARIMAX fits succeeded. Try increasing BIN_MINUTES_ARIMAX and/or lowering MIN_N_PER_PATIENT.")

# ----------------------------
# Summaries
# ----------------------------
print("\n=== Per-patient ARIMAX synchrony betas ===")
print(arimax_df[["n","beta","se","phi_ar1"]].describe().to_string())

stats = across_patient_tests(arimax_df)

print("\n=== Across-patient aggregation (patient is the unit) ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

# Optional: show how many patients have negative beta
frac_neg = np.mean(arimax_df["beta"].values < 0)
print(f"\nFraction of patients with beta<0: {frac_neg:.3f} ({(arimax_df['beta']<0).sum()}/{len(arimax_df)})")

# ----------------------------
# Save outputs
# ----------------------------
out_csv = os.path.join(out_dir, f"arimax_ar1_per_patient_binned{BIN_MINUTES_ARIMAX}_{COHORT}.csv")
arimax_df.to_csv(out_csv, index=False)

out_txt = os.path.join(out_dir, f"arimax_ar1_summary_binned{BIN_MINUTES_ARIMAX}_{COHORT}.txt")
with open(out_txt, "w") as f:
    f.write(f"COHORT={COHORT}\n")
    f.write(f"BIN_MINUTES_ARIMAX={BIN_MINUTES_ARIMAX}\n")
    f.write(f"MIN_N_PER_PATIENT={MIN_N_PER_PATIENT}\n")
    f.write(f"engine={'SARIMAX' if use_sarimax else 'ARIMA'}\n")
    f.write(f"fits_succeeded={len(arimax_df)}\n")
    f.write(f"fits_failed={n_fail}\n\n")
    f.write("Per-patient betas summary:\n")
    f.write(arimax_df[["n","beta","se","phi_ar1"]].describe().to_string())
    f.write("\n\nAcross-patient aggregation:\n")
    for k, v in stats.items():
        f.write(f"{k}: {v}\n")

print(f"\nSaved:")
print(f"  {out_csv}")
print(f"  {out_txt}")



Per-patient ARIMAX(AR1) on COHORT=full
Data: 30-min binned
Rows: 7900  Patients: 67

Fits succeeded: 54  |  failed: 0

=== Per-patient ARIMAX synchrony betas ===
                n       beta         se    phi_ar1
count   54.000000  54.000000  54.000000  54.000000
mean   141.574074  -0.011162   0.174198   0.836525
std     80.308039   0.123710   0.207886   0.243271
min     40.000000  -0.392689   0.000036  -0.535698
25%     77.500000  -0.043076   0.020745   0.775285
50%    119.500000  -0.001863   0.094257   0.904751
75%    187.250000   0.007102   0.252692   0.972680
max    378.000000   0.517056   1.014489   1.043464

=== Across-patient aggregation (patient is the unit) ===
  n_patients_used: 54
  frac_negative: 0.6111111111111112
  t_test_p: 0.5101918984007551
  sign_test_p: 0.13367423536410522
  meta_beta_ivw: 7.506399031729783e-05
  meta_se_ivw: 2.5256033175350952e-05
  meta_p_ivw: 0.0029574995909256163

Fraction of patients with beta<0: 0.611 (33/54)

Saved:
  ../../cnt-data/pi_notebo

## Analysis 1: Patient-clustered robust inference

We re-estimated the synchrony-ASM association using generalized estimating equations (GEE) with the patient as the clustering unit and robust standard errors. Robust standard errors are specifically designed to produce valid confidence estimates regardless of how correlated the measurements are within each patient. Under this approach, synchrony remained significantly negatively associated with medication load (β = −0.153, robust SE = 0.072, P = 0.033; 67 clusters). This directly tests the referee’s concern: even when statistical confidence is computed in a way that makes no assumptions about within-patient independence, the result holds.

In [5]:
gee_ind = fit_gee(df, Independence())
print(gee_ind.summary())

with open(os.path.join(out_dir, "gee_independence_robust_summary.txt"), "w") as f:
    f.write(str(gee_ind.summary()))

print("\nSynchrony (GEE-Ind robust):")
print("  beta =", float(gee_ind.params[SYNC]))
print("  se   =", float(gee_ind.bse[SYNC]))
print("  p    =", float(gee_ind.pvalues[SYNC]))

print("\nSaved:", os.path.join(out_dir, "gee_independence_robust_summary.txt"))


                                 GEE Regression Results                                
Dep. Variable:        med_sum_no_lorazepam_raw   No. Observations:                23402
Model:                                     GEE   No. clusters:                       67
Method:                            Generalized   Min. cluster size:                   1
                          Estimating Equations   Max. cluster size:                1128
Family:                               Gaussian   Mean cluster size:               349.3
Dependence structure:             Independence   Num. iterations:                     2
Date:                         Thu, 09 Apr 2026   Scale:                           2.649
Covariance type:                        robust   Time:                         20:41:53
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept               

## Analysis 2: Dependence-preserving permutation test

We performed a within-patient circular time-shift permutation test with 300 permutations. This procedure takes each patient’s synchrony time series and shifts it in time (wrapping around) so that the correlation structure within the patient is perfectly preserved, but the alignment between synchrony and ASM load is broken. If the referee’s hypothesis were correct—that significance is merely an artifact of correlated windows—then this shuffled data should produce results just as “significant” as the real data. It did not: the observed association was stronger than all but 1 of the 300 shuffled datasets (empirical two-sided P = 0.003).

In [6]:
N_PERM = 300
obs_beta, perm_betas, p_emp = circular_shift_permutation(df, n_perm=N_PERM, seed=0, min_shift=10)

print(f"Permutation fits used: {len(perm_betas)}/{N_PERM}")
print("Observed beta:", obs_beta)
print("Perm beta mean (sd):", float(np.mean(perm_betas)), float(np.std(perm_betas)))
print("Empirical two-sided p (|beta_perm| >= |beta_obs|):", p_emp)

out_csv = os.path.join(out_dir, f"perm_betas_circular_shift_{COHORT}.csv")
pd.DataFrame({"beta_perm": perm_betas}).to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


Permutation fits used: 300/300
Observed beta: -0.15254180089370142
Perm beta mean (sd): -0.13631474422843948 0.006956791443453073
Empirical two-sided p (|beta_perm| >= |beta_obs|): 0.0033333333333333335

Saved: ../../cnt-data/pi_notebook_new_analyses/full/perm_betas_circular_shift_full.csv


## Analysis 3: Likelihood ratio testing

We compared our statistical model with and without the synchrony term to ask: does including synchrony measurably improve the model’s ability to explain medication load, above and beyond all other variables? Using standard likelihood ratio tests, the answer was yes: both with a simpler model structure (likelihood ratio = 328.3, P < 10⁻⁷³) and with a more complex model that allows the synchrony effect to vary across patients (likelihood ratio = 9.57, P=0.002). This confirms that synchrony contributes genuine explanatory power, not just statistical noise amplified by repeated measurements.

In [7]:
# LRT: MixedLM with vs without synchrony (ML fits)

import statsmodels.formula.api as smf
from scipy.stats import chi2

def drop_term_from_formula(formula, term):
    # assumes formula like "Y ~ a + b + c"
    lhs, rhs = formula.split("~", 1)
    terms = [t.strip() for t in rhs.split("+")]
    terms = [t for t in terms if t != term]
    if len(terms) == 0:
        rhs2 = "1"
    else:
        rhs2 = " + ".join(terms)
    return lhs.strip() + " ~ " + rhs2

def fit_mixedlm_ml(formula, data, groups, re_formula=None, method="lbfgs"):
    m = smf.mixedlm(formula, data=data, groups=groups, re_formula=re_formula)
    r = m.fit(reml=False, method=method)
    return r

def lrt(full_res, red_res, label="LRT"):
    llf_full = float(full_res.llf)
    llf_red  = float(red_res.llf)
    stat = 2.0 * (llf_full - llf_red)

    # df_modelwc includes fixed + variance/cov parameters
    df_full = int(full_res.df_modelwc)
    df_red  = int(red_res.df_modelwc)
    df_diff = df_full - df_red

    p = float(chi2.sf(stat, df_diff)) if df_diff > 0 else float("nan")

    out = {
        "label": label,
        "llf_full": llf_full,
        "llf_reduced": llf_red,
        "lr_stat": stat,
        "df_diff": df_diff,
        "p_value": p,
        "aic_full": float(full_res.aic),
        "aic_reduced": float(red_res.aic),
        "bic_full": float(full_res.bic),
        "bic_reduced": float(red_res.bic),
        "converged_full": bool(getattr(full_res, "converged", True)),
        "converged_reduced": bool(getattr(red_res, "converged", True)),
    }
    return out

# ----------------------------
# Build reduced formula (drop synchrony fixed effect)
# ----------------------------
FORMULA_NO_SYNC = drop_term_from_formula(FORMULA, SYNC)

print("Full formula:     ", FORMULA)
print("Reduced formula:  ", FORMULA_NO_SYNC)

# =============================================================================
# A) Random-intercept only: LRT for adding synchrony as a fixed effect
# =============================================================================
print("\n====================")
print("A) Random-intercept MixedLM (ML): with vs without synchrony")
print("====================")

res_full_ri = fit_mixedlm_ml(FORMULA, df, groups=df[GROUP], re_formula=None)
res_red_ri  = fit_mixedlm_ml(FORMULA_NO_SYNC, df, groups=df[GROUP], re_formula=None)

print("\n--- Full (RI) ---")
print(res_full_ri.summary())
print("\n--- Reduced (RI; no synchrony fixed effect) ---")
print(res_red_ri.summary())

lrt_ri = lrt(res_full_ri, res_red_ri, label="RI: add synchrony fixed effect")
print("\nLRT result (RI):")
for k, v in lrt_ri.items():
    print(f"  {k}: {v}")

# =============================================================================
# B) Random intercept + random slope (Synchrony): LRT for fixed synchrony effect
#    (keeps random-effects structure identical in both models)
# =============================================================================
print("\n====================")
print("B) Random-slope MixedLM (ML): fixed synchrony effect test (same RE in both)")
print("====================")

re_form = f"1 + {SYNC}"

# Full: synchrony in fixed + random slope
res_full_rs = fit_mixedlm_ml(FORMULA, df, groups=df[GROUP], re_formula=re_form)

# Reduced: remove synchrony from fixed, but KEEP random slope for synchrony
# (This isolates the fixed-effect contribution while holding RE structure constant.)
res_red_rs  = fit_mixedlm_ml(FORMULA_NO_SYNC, df, groups=df[GROUP], re_formula=re_form)

print("\n--- Full (RI+RS) ---")
print(res_full_rs.summary())
print("\n--- Reduced (RI+RS; no synchrony fixed effect, but random slope kept) ---")
print(res_red_rs.summary())

lrt_rs = lrt(res_full_rs, res_red_rs, label="RI+RS: add synchrony fixed effect")
print("\nLRT result (RI+RS):")
for k, v in lrt_rs.items():
    print(f"  {k}: {v}")

print("\nDone.")


Full formula:      med_sum_no_lorazepam_raw ~ synchrony_broadband + time_since_last_seizure + time_until_next_seizure + emu_minute + ad_ratio + tod_sin + tod_cos + tod_sin2 + tod_cos2 + spike_rate
Reduced formula:   med_sum_no_lorazepam_raw ~ time_since_last_seizure + time_until_next_seizure + emu_minute + ad_ratio + tod_sin + tod_cos + tod_sin2 + tod_cos2 + spike_rate

A) Random-intercept MixedLM (ML): with vs without synchrony

--- Full (RI) ---
                Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: med_sum_no_lorazepam_raw
No. Observations: 23402   Method:             ML                      
No. Groups:       67      Scale:              0.4021                  
Min. group size:  1       Log-Likelihood:     -22784.9086             
Max. group size:  1128    Converged:          Yes                     
Mean group size:  349.3                                               
--------------------------------------------------------------------

## Analysis 4: Flexible correlation structure check

As a further complementary check, we aggregated data into 30-minute bins to reduce the number of measurements per patient and fitted a model with a flexible correlation structure that does not impose any particular pattern on how measurements relate to each other over time. Synchrony remained significantly negative (β = −0.028, P = 0.007).

In [8]:
# Notes:
# - "Unstructured" estimates a free correlation matrix within each cluster.
# - This typically requires *balanced* clusters (same number of time points per patient),
#   and can be computationally heavy as params scale ~ T*(T-1)/2.
# - To make it stable and comparable across patients, we:
#     (1) optionally bin within patient
#     (2) enforce a fixed number of timepoints per patient (take first T per patient)

import statsmodels.api as sm

# ----------------------------
# Settings
# ----------------------------
BIN_MINUTES_FOR_UNSTRUCT = 30   # use 30-min bins to reduce autocorrelation + size; set None for unbinned
T_PER_PATIENT = 20             # fixed number of (binned) timepoints per patient for unstructured correlation
MIN_PATIENTS_REQUIRED = 5
PRINT_SUMMARY = True

# ----------------------------
# Import cov_struct.Unstructured (with graceful fallback)
# ----------------------------
try:
    from statsmodels.genmod.cov_struct import Unstructured
    HAS_UNSTRUCTURED = True
except Exception:
    HAS_UNSTRUCTURED = False

# If Unstructured isn't available in your statsmodels build, fallback suggestion:
try:
    from statsmodels.genmod.cov_struct import Stationary
    HAS_STATIONARY = True
except Exception:
    HAS_STATIONARY = False

# ----------------------------
# Prepare data (optional binning)
# ----------------------------
if BIN_MINUTES_FOR_UNSTRUCT is None:
    d0 = df.copy()
    label = "unbinned"
else:
    d0 = bin_within_patient(df, BIN_MINUTES_FOR_UNSTRUCT)
    label = f"binned{BIN_MINUTES_FOR_UNSTRUCT}min"

# enforce ordering and rebuild time_index (safety)
d0 = d0.sort_values([GROUP, TIME_RAW]).reset_index(drop=True)
d0["time_index"] = d0.groupby(GROUP).cumcount().astype(int)

# ----------------------------
# Balance to fixed T_PER_PATIENT time points per patient
# ----------------------------
sizes = d0.groupby(GROUP).size()
eligible = sizes[sizes >= T_PER_PATIENT].index.values
d1 = d0[d0[GROUP].isin(eligible)].copy()

if d1[GROUP].nunique() < MIN_PATIENTS_REQUIRED:
    raise RuntimeError(
        f"Not enough patients with >= {T_PER_PATIENT} timepoints "
        f"(found {d1[GROUP].nunique()}). Reduce T_PER_PATIENT or bin more coarsely."
    )

# take first T per patient (balanced panel)
d1 = (d1.sort_values([GROUP, TIME_RAW])
        .groupby(GROUP, as_index=False, group_keys=False)
        .head(T_PER_PATIENT)
        .reset_index(drop=True))

# time_index must be 0..T-1 within each patient
d1["time_index"] = d1.groupby(GROUP).cumcount().astype(int)

print("\n====================")
print(f"GEE Unstructured on {COHORT} ({label})")
print(f"Patients used (>=T): {d1[GROUP].nunique()} / {df[GROUP].nunique()}")
print(f"Rows used: {len(d1)}  (T_PER_PATIENT={T_PER_PATIENT})")
print("====================")

# ----------------------------
# Fit
# ----------------------------
if HAS_UNSTRUCTURED:
    cov_struct = Unstructured()
    cov_label = "Unstructured"
else:
    if not HAS_STATIONARY:
        raise RuntimeError(
            "statsmodels in this environment does not expose cov_struct.Unstructured (or Stationary).\n"
            "Upgrade statsmodels or use Exchangeable/AR(1)/Stationary instead."
        )
    cov_struct = Stationary()
    cov_label = "Stationary (fallback; lag-based correlation)"

m = smf.gee(
    formula=FORMULA,
    groups=d1[GROUP],
    data=d1,
    family=Gaussian(),
    cov_struct=cov_struct,
    time=d1["time_index"].values,
)

res = m.fit(cov_type="robust")

print(f"\nWorking correlation: {cov_label}")
rho_info = getattr(res.cov_struct, "dep_params", None)
if rho_info is not None:
    # can be scalar or array depending on structure
    try:
        rho_flat = np.asarray(rho_info).ravel()
        print(f"dep_params shape={rho_flat.shape}, first few={rho_flat[:min(10,len(rho_flat))]}")
    except Exception:
        print(f"dep_params={rho_info}")

if PRINT_SUMMARY:
    print(res.summary())

beta_sync = float(res.params[SYNC])
se_sync = float(res.bse[SYNC])
p_sync = float(res.pvalues[SYNC])

print(f"\nSynchrony ({SYNC}) under {cov_label}:")
print(f"  beta = {beta_sync:.6f}")
print(f"  se   = {se_sync:.6f}")
print(f"  p    = {p_sync:.6g}")

# ----------------------------
# Save outputs
# ----------------------------
out_txt = os.path.join(out_dir, f"gee_{cov_label.replace(' ','_').replace('(','').replace(')','')}_{label}_T{T_PER_PATIENT}_summary.txt")
out_csv = os.path.join(out_dir, f"gee_{cov_label.replace(' ','_').replace('(','').replace(')','')}_{label}_T{T_PER_PATIENT}_params.csv")

with open(out_txt, "w") as f:
    f.write(f"COHORT={COHORT}\n")
    f.write(f"label={label}\n")
    f.write(f"cov_struct={cov_label}\n")
    f.write(f"T_PER_PATIENT={T_PER_PATIENT}\n")
    f.write(f"patients_used={d1[GROUP].nunique()}\n")
    f.write(f"rows_used={len(d1)}\n\n")
    f.write(res.summary().as_text())

pd.DataFrame({
    "term": res.params.index,
    "beta": res.params.values,
    "se": res.bse.values,
    "p": res.pvalues.values
}).to_csv(out_csv, index=False)

print(f"\nSaved:")
print(f"  {out_txt}")
print(f"  {out_csv}")



GEE Unstructured on full (binned30min)
Patients used (>=T): 59 / 67
Rows used: 1180  (T_PER_PATIENT=20)

Working correlation: Unstructured
dep_params shape=(400,), first few=[1.         0.99747887 0.98499896 0.97992162 0.97807197 0.9764201
 0.97578575 0.97512944 0.97085758 0.97159124]
                                 GEE Regression Results                                
Dep. Variable:        med_sum_no_lorazepam_raw   No. Observations:                 1180
Model:                                     GEE   No. clusters:                       59
Method:                            Generalized   Min. cluster size:                  20
                          Estimating Equations   Max. cluster size:                  20
Family:                               Gaussian   Mean cluster size:                20.0
Dependence structure:             Unstructured   Num. iterations:                    51
Date:                         Thu, 09 Apr 2026   Scale:                           2.234
Covarianc